# Brain Tumor Classification: Enhanced Single-Modal Training

## Environment Initialization


In [ ]:
import sys
import os
import time
from pathlib import Path

# ==================== Project Path Setup ====================
project_root = os.getcwd()
if project_root not in sys.path:
 sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

# ==================== Import Training Modules ====================
from src.training.enhanced_single_modal_trainer import EnhancedSingleModalTrainer
from src.core.utils import save_json, get_device, set_seed, TrainingLogger

# ==================== GPU Availability Check ====================
import torch
device = get_device()
print(f"Compute device: {device}")
if torch.cuda.is_available():
 print(f" GPU model: {torch.cuda.get_device_name(0)}")
 print(f" GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
 print(" Warning: No GPU detected. Training will be slow.")

print("\nEnvironment initialization complete.")


## Random Seed Configuration


In [ ]:
# ==================== Random Seed Configuration ====================
# Random seed affects:
# - Model parameter initialization
# - Training data shuffling
# - Dropout and other stochastic operations

RANDOM_SEED = 41 # Default: 41

# Apply random seed to all libraries
set_seed(RANDOM_SEED)

print(f"Random seed set to: {RANDOM_SEED}")
print("Experimental results are now fully reproducible")
print("Note: Different seed values produce different but reproducible results")


## Training Configuration


In [ ]:
# ==================== Data Configuration ====================
# Expected directory structure:
# data/train/{class_name}/image.jpg
# data/test/{class_name}/image.jpg

DATA_TRAIN_PATH = 'data/train'
DATA_TEST_PATH = 'data/test'

# ==================== Training Parameters ====================

NUM_EPOCHS = 40 # Maximum epochs (subject to early stopping)
PATIENCE = 10 # Early stopping patience

# ==================== Hyperparameter Grid Search ====================
#[0.001, 0.0001, 0.00001, 0.000001]
LEARNING_RATES = [ 0.0001] # LR candidates
OPTIMIZERS = ['''SGD''', 'Adam'] # Optimizer algorithms
WEIGHT_DECAY = 1e-4 # L2 regularization
SCHEDULER = 'CosineAnnealingLR' # LR scheduler





# ==================== Model Selection ====================
# Available architectures - comment out models to skip evaluation

MODEL_NAMES = [
 'EfficientNet_b0',
 'ResNet50',
 'MobileNetV3_large',
 'ViT',
 'DenseNet121',
 'DeiT',
 'Swin Transformer',
 'MambaOut'
]

# Batch size per model (adjust based on GPU memory)
BATCH_SIZE_CONFIG = {
 #'Swin Transformer': 32,
 #'MambaOut': 32,
 #'ViT': 32,
 #'DeiT': 32,
 #'EfficientNet_b0': 32,
 #'MobileNetV3_large': 32,
 #'ResNet50': 32,
 'DenseNet121': 32
}

# ==================== System Configuration ====================

NUM_WORKERS = 4 # DataLoader workers
MAX_RETRIES = 3 # Training retry attempts
RETRY_DELAY = 10 # Retry delay (seconds)

# Output configuration
SAVE_PLOTS = True
PLOT_DPI = 600
RESULTS_FILENAME = 'results/training_logs/enhanced_single_modal_results.json'

# ==================== Configuration Summary ====================

print("Configuration:")
print(f" Epochs: {NUM_EPOCHS}, Patience: {PATIENCE}")
print(f" Learning rates: {LEARNING_RATES}")
print(f" Optimizers: {OPTIMIZERS}")
print(f" Weight decay: {WEIGHT_DECAY}")
print(f"\nExperiment scale:")
print(f" Models: {len(MODEL_NAMES)}")
print(f" Configurations per model: {len(OPTIMIZERS)} × {len(LEARNING_RATES)} = {len(OPTIMIZERS) * len(LEARNING_RATES)}")
print(f" Total experiments: {len(MODEL_NAMES) * len(OPTIMIZERS) * len(LEARNING_RATES)}")
print(f"\nOutput:")
print(f" Results: {RESULTS_FILENAME}")
print(f" Checkpoints: results/best_models/")
print(f" Visualization: {'Enabled' if SAVE_PLOTS else 'Disabled'}")


## Dataset Verification


In [ ]:
print("Verifying dataset...\n")

# ==================== Train Set Verification ====================
if os.path.exists(DATA_TRAIN_PATH):
 print(f"Train path: {DATA_TRAIN_PATH}")
 train_classes = [d for d in os.listdir(DATA_TRAIN_PATH) if os.path.isdir(os.path.join(DATA_TRAIN_PATH, d))]
 print(f" Classes: {len(train_classes)}")
 print(f" Names: {train_classes}")
 
 print(f" Distribution:")
 total_train = 0
 for cls in train_classes:
 cls_path = os.path.join(DATA_TRAIN_PATH, cls)
 num_samples = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
 print(f" - {cls}: {num_samples}")
 total_train += num_samples
 print(f" Total: {total_train} images")
else:
 print(f"Error: Train path not found: {DATA_TRAIN_PATH}")
 raise FileNotFoundError(f"Training data not found: {DATA_TRAIN_PATH}")

print()

# ==================== Test Set Verification ====================
if os.path.exists(DATA_TEST_PATH):
 print(f"Test path: {DATA_TEST_PATH}")
 test_classes = [d for d in os.listdir(DATA_TEST_PATH) if os.path.isdir(os.path.join(DATA_TEST_PATH, d))]
 print(f" Classes: {len(test_classes)}")
 print(f" Names: {test_classes}")
 
 print(f" Distribution:")
 total_test = 0
 for cls in test_classes:
 cls_path = os.path.join(DATA_TEST_PATH, cls)
 num_samples = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
 print(f" - {cls}: {num_samples}")
 total_test += num_samples
 print(f" Total: {total_test} images")
else:
 print(f"Error: Test path not found: {DATA_TEST_PATH}")
 raise FileNotFoundError(f"Test data not found: {DATA_TEST_PATH}")

print("\nDataset verification complete.")


## Configuration Object


In [ ]:
# ==================== Configuration Class ====================
class TrainingConfig:
 def __init__(self):
 # Training parameters
 self.NUM_EPOCHS = NUM_EPOCHS
 self.LEARNING_RATES = LEARNING_RATES
 self.OPTIMIZERS = OPTIMIZERS
 self.WEIGHT_DECAY = WEIGHT_DECAY
 self.SCHEDULER = SCHEDULER
 
 # Data paths
 self.DATA_TRAIN_PATH = DATA_TRAIN_PATH
 self.DATA_TEST_PATH = DATA_TEST_PATH
 
 # System configuration
 self.NUM_WORKERS = NUM_WORKERS
 self.BATCH_SIZE_CONFIG = BATCH_SIZE_CONFIG
 self.MAX_RETRIES = MAX_RETRIES
 self.RETRY_DELAY = RETRY_DELAY
 
 # Output configuration
 self.SAVE_PLOTS = SAVE_PLOTS
 self.PLOT_DPI = PLOT_DPI
 self.JSON_FILENAME = RESULTS_FILENAME

config = TrainingConfig()
print("Configuration object created.")


## Trainer Initialization


In [ ]:
print("Initializing training system...\n")

# Create trainer instance
trainer = EnhancedSingleModalTrainer(config)

print("\nTraining system ready.")


## Logger Initialization


In [ ]:
# ==================== Logger Initialization ====================
from datetime import datetime

# Generate timestamped experiment name
experiment_name = f"enhanced_single_modal_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
logger = TrainingLogger(log_dir='logs', experiment_name=experiment_name)

# Log configuration parameters
config_dict = {
 'random_seed': RANDOM_SEED,
 'max_epochs': NUM_EPOCHS,
 'early_stopping_patience': PATIENCE,
 'learning_rates': LEARNING_RATES,
 'optimizers': OPTIMIZERS,
 'batch_size': BATCH_SIZE_CONFIG,
 'num_workers': NUM_WORKERS,
 'weight_decay': WEIGHT_DECAY,
 'lr_scheduler': SCHEDULER
}
logger.log_config(config_dict)

print(f"Logger initialized")
print(f" Experiment: {experiment_name}")
print(f" Log directory: logs/")
print(f" Text log: logs/{experiment_name}.log")
print(f" JSON log: logs/{experiment_name}_detailed.json")
print("\nTraining process will be fully logged.")


## Training Execution


In [ ]:
# ==================== Log Experiment Start ====================
logger.log_message("Starting deep learning model training experiment")
logger.log_message("=" * 80)
logger.log_message("Automated features:")
logger.log_message(f" Early stopping (patience={PATIENCE} epochs)")
logger.log_message(" Auto-save best models (only optimal weights per model)")
logger.log_message(" Training time and performance metrics logging")
logger.log_message(" Detailed training statistics generation")
logger.log_message("=" * 80)

print("Training started. Please wait...")
print(f"Models to train: {', '.join(MODEL_NAMES)}")
print(f"Configurations per model: {len(OPTIMIZERS) * len(LEARNING_RATES)}")
print("=" * 80)
print()

# Record experiment start time
experiment_start_time = time.time()

try:
 # ==================== Execute Training ====================
 # Automated steps:
 # 1. Grid search for each model
 # 2. Early stopping to prevent overfitting
 # 3. Save best model checkpoints
 # 4. Log all training data
 results = trainer.run_enhanced_grid_search(
 resume_from_checkpoint=False, # False=start fresh, True=resume from checkpoint
 patience=PATIENCE
 )
 
 # ==================== Log Training Results ====================
 if results:
 for result in results:
 if result['best_acc'] > 0:
 best_config = result['best_config']
 logger.log_model_result(
 model_name=result['name'],
 optimizer=best_config['optimizer'],
 lr=best_config['lr'],
 result={
 'best_val_acc': result['best_acc'],
 'best_epoch': best_config['best_epoch'],
 'time_to_best': best_config['time_to_best'],
 'early_stopped': best_config['early_stopped'],
 'best_model_path': f"results/best_models/{result['name'].replace(' ', '_')}_{best_config['optimizer']}_lr{best_config['lr']}_best.pth"
 }
 )
 
 print("\n" + "=" * 80)
 print("Training completed successfully")
 print("=" * 80)
 
except KeyboardInterrupt:
 print("\nTraining interrupted by user")
 print("Completed models have been saved. Results available below.")
 results = None
 
except Exception as e:
 print(f"\nTraining error: {e}")
 print("If out of memory, try reducing BATCH_SIZE")
 import traceback
 traceback.print_exc()
 results = None

# Calculate total time
total_experiment_time = time.time() - experiment_start_time

# Save experiment summary
logger.log_experiment_summary(total_experiment_time)

# Display statistics
print(f"\nTotal time: {total_experiment_time/3600:.2f} hours ({total_experiment_time/60:.1f} minutes)")
print(f"\nComplete logs saved to:")
print(f" Text format: logs/{experiment_name}.log")
print(f" JSON format: logs/{experiment_name}_detailed.json")
print(f"\nBest model checkpoints: results/best_models/")
print(f"Training results: {RESULTS_FILENAME}")


## Training Results Analysis


In [ ]:
# ==================== Analyze and Display Training Results ====================
if results is not None and len(results) > 0:
 print("\n" + "=" * 80)
 print("Training Results Summary")
 print("=" * 80)
 
 # Filter successful models
 successful_results = [r for r in results if r['best_acc'] > 0]
 
 if successful_results:
 print(f"\nSuccessfully trained: {len(successful_results)}/{len(results)} models\n")
 
 # Statistics
 total_time_to_best = 0
 total_saved_time = 0
 
 # ==================== Display Detailed Results Per Model ====================
 for i, result in enumerate(successful_results, 1):
 best_config = result['best_config']
 time_to_best = best_config['time_to_best']
 early_stopped = best_config['early_stopped']
 best_epoch = best_config['best_epoch']
 
 total_time_to_best += time_to_best
 
 # Calculate time saved by early stopping
 if early_stopped:
 saved_epochs = NUM_EPOCHS - (best_epoch + 1)
 estimated_saved_time = (time_to_best / (best_epoch + 1)) * saved_epochs
 total_saved_time += estimated_saved_time
 early_stop_info = f"Early stopped, saved ~{estimated_saved_time/60:.1f} min"
 else:
 early_stop_info = "Completed full training"
 
 print(f"[{i}] {result['name']}")
 print(f" Best val acc: {result['best_acc']*100:.2f}% (epoch {best_epoch+1})")
 print(f" Best config: {best_config['optimizer']}, LR={best_config['lr']}")
 print(f" Training time: {time_to_best/60:.2f} min")
 print(f" Status: {early_stop_info}")
 print(f" Checkpoint: results/best_models/{result['name'].replace(' ', '_')}_{best_config['optimizer']}_lr{best_config['lr']}_best.pth")
 print()
 
 # ==================== Find Best Performing Model ====================
 best_result = max(successful_results, key=lambda x: x['best_acc'])
 print("\n" + "=" * 80)
 print("Top Performing Model")
 print("=" * 80)
 print(f"Model: {best_result['name']}")
 print(f"Best accuracy: {best_result['best_acc']*100:.2f}%")
 print(f"Best config: {best_result['best_config']['optimizer']}, LR={best_result['best_config']['lr']}")
 print(f"Parameters: {best_result['param_count']:,}")
 print(f"Checkpoint: results/best_models/{best_result['name'].replace(' ', '_')}_{best_result['best_config']['optimizer']}_lr{best_result['best_config']['lr']}_best.pth")
 
 # ==================== Training Efficiency Statistics ====================
 print("\n" + "=" * 80)
 print("Training Efficiency Analysis")
 print("=" * 80)
 print(f"Total time: {total_experiment_time/60:.1f} min ({total_experiment_time/3600:.2f} hours)")
 print(f"Effective training time: {total_time_to_best/60:.1f} min (to reach best accuracy)")
 if total_saved_time > 0:
 print(f"Time saved by early stopping: ~{total_saved_time/60:.1f} min")
 print(f"Efficiency gain: {(total_saved_time/total_experiment_time)*100:.1f}%")
 
 print(f"\nOutput files:")
 print(f" Results: {RESULTS_FILENAME}")
 print(f" Checkpoints: results/best_models/")
 print(f" Logs: logs/")
 
 else:
 print("All model training failed. Please check data and configuration.")
else:
 print("No training results available (training may have been interrupted or not executed)")


## Hyperparameter Analysis Visualization


In [ ]:
# ==================== Generate Hyperparameter Analysis Plots ====================
import os
import json
from pathlib import Path
from src.visualization.plotting import ComparisonPlotter

# Read training results file
results_file = "results/training_logs/enhanced_single_modal_results.json"

if not os.path.exists(results_file):
 print(f"Warning: Training results file not found: {results_file}")
 print("Please run complete training before viewing hyperparameter analysis")
else:
 print(f"Reading training results: {results_file}...")
 
 try:
 with open(results_file, 'r', encoding='utf-8') as f:
 results = json.load(f)
 
 # Filter valid data (exclude incomplete training)
 valid_results = [r for r in results if r.get('best_acc', 0) > 0 and r.get('optimizer_lr_results')]
 
 print(f"Successfully loaded training data for {len(valid_results)} models\n")
 
 if valid_results:
 print("Preparing hyperparameter analysis plots...")
 print("=" * 80)
 
 # Display data summary
 for r in valid_results:
 num_configs = len(r['optimizer_lr_results'])
 print(f" {r['name']}: {num_configs} hyperparameter configs")
 print(f" Best val accuracy: {r['best_acc']*100:.2f}%")
 if r.get('best_config'):
 print(f" Best config: {r['best_config']['optimizer']} lr={r['best_config']['lr']}")
 print()
 
 print("=" * 80)
 print()
 
 # Create plotter and generate plots
 try:
 plotter = ComparisonPlotter(save_dir="results/plots", dpi=600)
 plotter.plot_hyperparameter_analysis(valid_results)
 
 print("Hyperparameter analysis plots generated!")
 print(f"Saved to: results/plots/analysis/hyperparameter_analysis.png")
 print()
 print("Plot description:")
 print(" - X-axis: Hyperparameter combinations (optimizer-learning_rate)")
 print(" - Y-axis: Validation accuracy (%)")
 print(" - Orange: Best configuration for each model")
 print(" - Layout: Max 4 models per row, auto-wrap")
 
 except Exception as e:
 print(f"Plotting failed: {e}")
 import traceback
 traceback.print_exc()
 else:
 print("Warning: No valid training data found")
 print("Please ensure at least one model trained successfully")
 
 except Exception as e:
 print(f"Failed to read results file: {e}")



## Test Set Evaluation of Best Model


In [ ]:
# Evaluate the best saved model on the test set and print accuracy
# This cell includes evaluation prep adapted from `Model_Evaluation_Visualization.ipynb` (data path discovery + test loader creation)
# Also extracts embeddings for t-SNE visualization
import os
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

from src.data.data_loaders import DataLoaderFactory

# --- Locate test directory (prefer DATA_TEST_PATH if present) ---
possible_data_roots = [
 "/root/autodl-tmp/classification/AI4BrainTumor/data",
 os.getcwd(),
 "data"
]

TEST_DIR = None
# First try notebook variable
if 'DATA_TEST_PATH' in globals() and os.path.exists(DATA_TEST_PATH):
 TEST_DIR = DATA_TEST_PATH
else:
 for root in possible_data_roots:
 test_path = os.path.join(root, "test")
 if os.path.exists(test_path):
 TEST_DIR = test_path
 break

if TEST_DIR is None:
 raise FileNotFoundError(f"Test directory not found. Checked: {possible_data_roots} and DATA_TEST_PATH")

print(f"Using test directory: {TEST_DIR}")

# --- Load training results to find best model checkpoint ---
results_file = "results/training_logs/enhanced_single_modal_results.json"
if not os.path.exists(results_file):
 raise FileNotFoundError(f"Results file not found: {results_file}")

with open(results_file, 'r', encoding='utf-8') as f:
 results = json.load(f)

# Select best model by recorded best_acc
best_entry = max(results, key=lambda r: r.get('best_acc', 0))
model_name = best_entry['name']
best_cfg = best_entry.get('best_config', {})
optimizer_name = best_cfg.get('optimizer')
lr = best_cfg.get('lr')

# Determine checkpoint path
ckpt_path = None
if optimizer_name is not None and lr is not None:
 key = f"{optimizer_name}_lr{lr}"
 opt_results = best_entry.get('optimizer_lr_results', {}).get(key, {})
 ckpt_path = opt_results.get('best_val_model_path') or opt_results.get('best_model_path')

if not ckpt_path:
 ckpt_path = best_entry.get('best_model_path')

if not ckpt_path:
 bm_dir = Path('results/best_models')
 candidates = sorted(bm_dir.glob('*_best*.pth'))
 if candidates:
 ckpt_path = str(candidates[-1])

if not ckpt_path:
 raise FileNotFoundError("Could not find a best model checkpoint path in results JSON or results/best_models/")

ckpt_path = Path(ckpt_path)
if not ckpt_path.exists():
 raise FileNotFoundError(f"Checkpoint file not found: {ckpt_path}")

print(f"Using checkpoint: {ckpt_path}")

# --- Create test DataLoader using the same transforms as training/validation ---
val_test_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)
# Use a reasonable batch size
batch_size = best_entry.get('batch_size', 32)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=NUM_WORKERS)
class_names = test_dataset.classes

print(f"Test set: {len(test_dataset)} samples | Classes: {class_names}")

# --- Build model and load weights robustly ---
num_classes = len(class_names)
model = trainer.create_model(model_name, num_classes)
checkpoint = torch.load(str(ckpt_path), map_location='cpu')

# Extract state dict
state_dict = None
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
 state_dict = checkpoint['model_state_dict']
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
 state_dict = checkpoint['state_dict']
elif isinstance(checkpoint, dict) and all(hasattr(v, 'dtype') for v in checkpoint.values()):
 state_dict = checkpoint
else:
 state_dict = checkpoint.get('model_state_dict') if isinstance(checkpoint, dict) else checkpoint

# Load state dict handling possible 'module.' prefix
try:
 model.load_state_dict(state_dict)
except Exception:
 new_state = { (k.replace('module.', '') if k.startswith('module.') else k): v for k, v in state_dict.items() }
 model.load_state_dict(new_state)

# Move to device and evaluate
model = model.to(device)
model.eval()
criterion = nn.CrossEntropyLoss()

# --- Extract embeddings for t-SNE ---
def extract_embeddings(model, data_loader, device, model_name):
 """
 Extract feature embeddings from the model on a dataset.
 Uses different extraction logic depending on the model architecture.
 """
 model.eval()
 all_embeddings = []
 all_labels = []
 
 with torch.no_grad():
 for inputs, labels in data_loader:
 inputs = inputs.to(device)
 
 # Choose extraction logic by model type
 if 'densenet' in model_name.lower():
 # DenseNet: use features layer + global average pooling
 features = model.features(inputs)
 embeddings = F.adaptive_avg_pool2d(features, (1, 1)).view(inputs.size(0), -1)
 elif hasattr(model, 'features'):
 # Other torchvision models (ResNet, EfficientNet, etc.)
 features = model.features(inputs)
 embeddings = F.adaptive_avg_pool2d(features, (1, 1)).view(inputs.size(0), -1)
 elif hasattr(model, 'avgpool') and hasattr(model, 'fc'):
 # ResNet family
 features = model.conv1(model.bn1(model.relu(model.maxpool(model.features(inputs)))))
 features = model.layer1(features)
 features = model.layer2(features)
 features = model.layer3(features)
 features = model.layer4(features)
 embeddings = model.avgpool(features).view(inputs.size(0), -1)
 elif hasattr(model, 'head') and hasattr(model.head, 'in_features'):
 # ViT, DeiT, etc.
 embeddings = model(inputs)
 else:
 # Default: use model output as embedding
 outputs = model(inputs)
 embeddings = outputs
 
 all_embeddings.append(embeddings.cpu().numpy())
 all_labels.extend(labels.numpy())
 
 all_embeddings = np.vstack(all_embeddings)
 all_labels = np.array(all_labels)
 
 return all_embeddings, all_labels

print(f"\nExtracting embeddings for t-SNE (model: {model_name})...")
embeddings, labels = extract_embeddings(model, test_loader, device, model_name)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Labels shape: {labels.shape}")

# Save embeddings and labels
os.makedirs('results/embeddings', exist_ok=True)
embedding_dir = Path('results/embeddings')

# Name the file using the model name and timestamp
timestamp = time.strftime("%Y%m%d_%H%M%S")
embedding_path = embedding_dir / f"{model_name}_test_embeddings.npz"

np.savez_compressed(embedding_path,
 embeddings=embeddings,
 labels=labels,
 class_names=json.dumps(class_names),
 model_name=model_name,
 checkpoint=str(ckpt_path))

print(f"\n Embeddings saved to: {embedding_path}")
print(f" - Embeddings: {embeddings.shape}")
print(f" - Labels: {labels.shape}")
print(f" - Classes: {class_names}")

# --- Evaluate model ---
test_loss, test_acc = trainer.evaluate_simplified(model, test_loader, criterion)
print(f"\nTest accuracy: {test_acc * 100:.2f}%")



In [ ]:
# DenseNet evaluation (record predictions, true labels, and embeddings) on additional_data/additional_mapped
import os
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

NEW_TEST_DIR = 'additional_data/additional_mapped'
if not os.path.exists(NEW_TEST_DIR):
 raise FileNotFoundError(f"New test directory not found: {NEW_TEST_DIR}")

val_test_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(NEW_TEST_DIR, transform=val_test_transform)
if len(test_dataset) == 0:
 raise RuntimeError(f"No images found under: {NEW_TEST_DIR}")

batch_size = BATCH_SIZE_CONFIG.get('DenseNet121', 32)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False, num_workers=NUM_WORKERS)
class_names = test_dataset.classes

# Find DenseNet checkpoint
bm_dir = Path('results/best_models')
dense_ckpts = sorted(bm_dir.glob('DenseNet121*_best*.pth'))
ckpt_path = None
if dense_ckpts:
 ckpt_path = str(dense_ckpts[-1])
else:
 results_file = 'results/training_logs/enhanced_single_modal_results.json'
 if os.path.exists(results_file):
 with open(results_file, 'r', encoding='utf-8') as f:
 results = json.load(f)
 for r in results:
 if r.get('name', '').lower().startswith('densenet'):
 best_model_path = r.get('best_model_path') or r.get('best_val_model_path')
 if best_model_path and os.path.exists(best_model_path):
 ckpt_path = best_model_path
 break

if not ckpt_path:
 raise FileNotFoundError('DenseNet checkpoint not found')

print(f"Using DenseNet checkpoint: {ckpt_path}")

# Load model - Force 6-class classifier to match training
num_classes = 6
model = trainer.create_model('DenseNet121', num_classes)
checkpoint = torch.load(ckpt_path, map_location='cpu')

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
 state_dict = checkpoint['model_state_dict']
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
 state_dict = checkpoint['state_dict']
else:
 state_dict = checkpoint if isinstance(checkpoint, dict) else checkpoint

try:
 model.load_state_dict(state_dict)
except Exception:
 new_state = { (k.replace('module.', '') if k.startswith('module.') else k): v for k, v in state_dict.items() }
 model.load_state_dict(new_state)

model = model.to(device)
model.eval()

# Lists for storing results
all_preds = []
all_trues = []
all_embeddings = []
all_probs = []
all_paths = []
correct = 0
total = 0

# Get file paths from dataset
file_paths = [p for p, _ in test_dataset.samples]
idx = 0

with torch.no_grad():
 for images, labels in test_loader:
 batch_size_curr = images.size(0)
 images, labels = images.to(device), labels.to(device)

 # Forward pass
 outputs = model(images)
 probs = F.softmax(outputs, dim=1)
 _, predicted = torch.max(outputs.data, 1)

 # Extract embeddings from DenseNet features layer
 # DenseNet.features output shape: [batch, 1024, H, W]
 features = model.features(images)
 # Global average pooling to get fixed-size feature vector [batch, 1024]
 embeddings = F.adaptive_avg_pool2d(features, (1, 1)).view(batch_size_curr, -1)

 # Store results
 all_preds.extend(predicted.cpu().tolist())
 all_trues.extend(labels.cpu().tolist())
 all_probs.extend(probs.cpu().tolist())
 all_embeddings.append(embeddings.cpu().numpy())

 batch_paths = file_paths[idx: idx + batch_size_curr]
 all_paths.extend(batch_paths)
 idx += batch_size_curr

 correct += (predicted == labels).sum().item()
 total += batch_size_curr

if total == 0:
 raise RuntimeError('No samples evaluated')

acc = correct / total
print(f"DenseNet Test accuracy on '{NEW_TEST_DIR}': {acc * 100:.2f}%")

# Use class names from training (6 classes)
model_class_names = ['NORMAL', 'Neurocitoma', 'Meningioma', 'Outros Tipos de Lesões', 'Glioma', 'Schwannoma']

# Save predictions and true labels
os.makedirs('results/collected_predictions', exist_ok=True)
out_json = 'results/collected_predictions/DenseNet_additional_mapped_preds.json'
with open(out_json, 'w', encoding='utf-8') as f:
 json.dump({
 'true_labels': all_trues,
 'predictions': all_preds,
 'probabilities': all_probs,
 'class_names': model_class_names,
 'accuracy': acc
 }, f, ensure_ascii=False, indent=2)

# Save embeddings for t-SNE
os.makedirs('results/embeddings', exist_ok=True)
out_npz = 'results/embeddings/DenseNet_additional_mapped_embeddings.npz'
emb_array = np.vstack(all_embeddings)
np.savez_compressed(out_npz,
 embeddings=emb_array,
 true_labels=np.array(all_trues),
 predictions=np.array(all_preds),
 file_paths=np.array(all_paths),
 class_names=model_class_names,
 checkpoint=str(ckpt_path))

print(f"Saved DenseNet predictions to: {out_json}")
print(f"Saved DenseNet embeddings to: {out_npz}")